In [60]:
import os

FILE_PATH = "../data/the-verdict.txt"

with open(FILE_PATH, "r", encoding="utf-8") as file:
    raw_text = file.read()


In [ ]:
from typing import Any

from tiktoken import Encoding
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer: Encoding, max_length = 10, stride = 10) -> None:
        self.input_tokens: list[list[int]] = []
        self.target_tokens: list[list[int]] = []

        tokens_ids = tokenizer.encode(text)

        for i in range(0, len(tokens_ids) - max_length, stride):
            self.input_tokens.append(tokens_ids[i:i + max_length])
            self.target_tokens.append(tokens_ids[i+1:i + max_length + 1])

    def __len__(self):
        return len(self.target_tokens)

    def __getitem__(self, index) -> Any:
        return self.input_tokens[index], self.target_tokens[index]

In [62]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt-2")
max_length = 10


dataset = GPTDataset(raw_text[:100], tokenizer, 10)

dataloader = DataLoader(dataset,batch_size=1, num_workers=0, shuffle=True, drop_last=True)


In [ ]:
from torch.nn import Embedding

n_vocab = tokenizer.n_vocab
embedding_dim = 5

token_embedding = Embedding(n_vocab, embedding_dim)

positional_embedding = Embedding(max_length, embedding_dim)